<a href="https://colab.research.google.com/github/asipnana/ProjectNLP/blob/main/notebooks/02_preprocessing_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [179]:
%cd /content/ProjectNLP

/content/ProjectNLP


In [180]:
!git pull

hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.


In [181]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [182]:
df = pd.read_csv("/content/ProjectNLP/dataset/raw/PRDECT-ID Dataset.csv")

In [183]:
df.head()

,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,Computers and Laptops,SSD Midasforce 128 Gb - Tanpa Caddy,Jakarta Barat,213000,5.0,12300,3573,5,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [184]:
df = df[['Customer Review', 'Sentiment', 'Emotion']]
df.head()

,Customer Review,Sentiment,Emotion
0,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [185]:
df.columns = ['review', 'sentiment', 'emotion']

In [186]:
df.isnull().sum()

,0
review,0
sentiment,0
emotion,0


In [187]:
df['review'].duplicated().sum()

np.int64(95)

In [188]:
df = df.drop_duplicates(subset=['review'])

In [189]:
# Data Cleaning TF-IDF dan FastText

def clean_text(text):
  text = str(text)
  text = text.lower()
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"www\S+", "", text)
  text = re.sub(r"@\w+", "", text)
  text = re.sub(r"#\w+", "", text)
  text = re.sub(r"[^a-zA-Z\s]", " ", text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()

In [190]:
df['clean_review'] = df['review'].apply(clean_text)

In [191]:
df[['review', 'clean_review']].head()

,review,clean_review
0,Alhamdulillah berfungsi dengan baik. Packaging...,alhamdulillah berfungsi dengan baik packaging ...
1,"barang bagus dan respon cepat, harga bersaing ...",barang bagus dan respon cepat harga bersaing d...
2,"barang bagus, berfungsi dengan baik, seler ram...",barang bagus berfungsi dengan baik seler ramah...
3,bagus sesuai harapan penjual nya juga ramah. t...,bagus sesuai harapan penjual nya juga ramah tr...
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",barang bagus pengemasan aman dapat berfungsi d...


In [192]:
sentiment_encoder = LabelEncoder()
df['sentiment_label'] = sentiment_encoder.fit_transform(df['sentiment'])

In [193]:
dict(zip(sentiment_encoder.classes_, range(len(sentiment_encoder.classes_))))

{'Negative': 0, 'Positive': 1}

In [194]:
emotion_encoder = LabelEncoder()
df['emotion_label'] = emotion_encoder.fit_transform(df['emotion'])

In [195]:
dict(zip(emotion_encoder.classes_, range(len(emotion_encoder.classes_))))

{'Anger': 0, 'Fear': 1, 'Happy': 2, 'Love': 3, 'Sadness': 4}

In [196]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['sentiment_label'])

In [197]:
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['sentiment_label'])

In [198]:
for name, data in[('Train', train_df), ('Val', val_df), ('Test', test_df)]:
  print(f"{name}")
  print(data['sentiment'].value_counts(normalize=True))
  print(data['emotion'].value_counts(normalize=True))

Train
sentiment
Negative    0.518718
Positive    0.481282
Name: proportion, dtype: float64
emotion
Happy      0.328037
Sadness    0.223000
Fear       0.172098
Love       0.153245
Anger      0.123620
Name: proportion, dtype: float64
Val
sentiment
Negative    0.518844
Positive    0.481156
Name: proportion, dtype: float64
emotion
Happy      0.353015
Sadness    0.212312
Fear       0.167085
Anger      0.139447
Love       0.128141
Name: proportion, dtype: float64
Test
sentiment
Negative    0.518844
Positive    0.481156
Name: proportion, dtype: float64
emotion
Happy      0.319095
Sadness    0.236181
Love       0.162060
Fear       0.150754
Anger      0.131910
Name: proportion, dtype: float64


In [199]:
SAVE_PATH = "/content/ProjectNLP/dataset/preprocessed/"

os.makedirs(SAVE_PATH, exist_ok=True)
train_df.to_csv(f"{SAVE_PATH}/train.csv", index=False)
val_df.to_csv(f"{SAVE_PATH}/val.csv", index=False)
test_df.to_csv(f"{SAVE_PATH}/test.csv", index=False)

In [201]:
!ls /content/ProjectNLP/dataset/preprocessed

test.csv  train.csv  val.csv
